In [1]:
import pandas as pd
import re
from ollama import chat, ChatResponse
from tqdm import tqdm  # 用於顯示進度條

In [2]:
# --- 設定 ---
FILE_PATH = './test/RACE_Middle_Combined.xlsx'     # 資料集路徑

In [3]:
# --- 函數：呼叫 Ollama (使用官方庫) ---
def query_ollama(prompt, model='gemma3:12b', context_len=1536, think=False):
    try:
        # 使用 ollama.chat 直接呼叫
        response: ChatResponse = chat(
            model=model,
            messages=[
                {'role': 'user', 'content': prompt}
            ],
            # 您可以在此加入 options，例如 temperature
            options={
                'num_ctx': context_len,  # 在這裡指定 Context Length
                'temperature': 0.0  # 讓輸出結果穩定
            },
            think=think  # 設定為 True 以獲取推理欄位
        )
        # 回傳模型的內容
        return response.message.content
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        return ""

In [4]:
# --- 函數：解析模型輸出 ---
def extract_answer(text):
    """
    從模型的回答中提取選項數字 (1, 2, 3, 4)。
    DeepSeek R1 會輸出 <think>...</think> 思考過程，需要過濾。
    """
    if not text:
        return None

    # 1. 移除 <think> 標籤內的內容 (DeepSeek R1 特有)
    # flags=re.DOTALL 確保 . 可以匹配換行符號
    text_cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    
    # 2. 策略 A: 尋找明確的 "Answer: X" 格式 (Prompt 中有要求)
    match = re.search(r'Answer:\s*([1-4])', text_cleaned, re.IGNORECASE)
    if match:
        return int(match.group(1))
    
    # 3. 策略 B: 如果沒有標準格式，尋找最後出現的數字 1-4
    # 這是一個備用方案
    numbers = re.findall(r'\b([1-4])\b', text_cleaned)
    if numbers:
        return int(numbers[-1])
        
    return None  # 解析失敗

In [5]:
# --- 主程式 ---
# 1. 讀取資料
print(f"正在讀取 {FILE_PATH}...")
try:
    df = pd.read_excel(FILE_PATH)
except Exception as e:
    print(f"讀取檔案失敗: {e}")

# 檢查必要的欄位
required_columns = ['ID', 'Article', 'Question', 'Option1', 'Option2', 'Option3', 'Option4', 'Answer']
if not all(col in df.columns for col in required_columns):
    print(f"錯誤: Excel 檔案缺少必要的欄位。需要: {required_columns}")

正在讀取 ./test/RACE_Middle_Combined.xlsx...


In [6]:
df

,ID,Original_ID,Article,Question,Option1,Option2,Option3,Option4,Answer
0,1,middle1.txt,"Take a class at Dulangkou School, and you'll s...",A discipline leader is supposed to _ .,take care of the whole group,make sure that everybody finishes homework,make sure that nobody chats in class,collect all the homework and hand it in to tea...,3
1,2,middle1.txt,"Take a class at Dulangkou School, and you'll s...",The new way of learning is said to give studen...,chat with each other,listen to the teacher,make friends,communicate,4
2,3,middle1.txt,"Take a class at Dulangkou School, and you'll s...",We can see from the story that some students ...,get benefits from,are tired of,cannot get used to,hate,1
3,4,middle1.txt,"Take a class at Dulangkou School, and you'll s...",How many leaders are there in one group?,Three.,Four.,Two.,Five or six.,1
4,5,middle1019.txt,"Before I left to meet Lynne, my friend told me...","After the meal, _ ,",Lynne paid only for herself,Lynne paid for both of us.,I would like to pay for myself,I paid for both of us.,1
...,...,...,...,...,...,...,...,...,...
1431,1432,middle999.txt,"My name is David and I have two brothers, Mark...",_ doesn't like French fries.,David,Mark,Bruce,David and Mark,3
1432,1433,middle999.txt,"My name is David and I have two brothers, Mark...",_ like hamburgers.,David and Mark,Mark and Bruce,"David, Mark, and Bruce",David and Bruce,3
1433,1434,middle999.txt,"My name is David and I have two brothers, Mark...",_ doesn't like eggs for breakfast.,Mark and Bruce,David,Mark,Bruce,2
1434,1435,middle999.txt,"My name is David and I have two brothers, Mark...",_ likes fruit for breakfast.,Mark and Bruce,David,Mark,Bruce,2


In [7]:
# --- 主程式 ---
def main():
    correct_count = 0
    total_questions = len(df)
    results = []

    print(f"開始使用 {MODEL_NAME} 處理 {total_questions} 個問題...")

    # 2. 遍歷每一列進行預測
    for index, row in tqdm(df.iterrows(), total=total_questions):
        article = row['Article']
        question = row['Question']
        # 建立選項字典方便取用
        options = {
            1: row['Option1'],
            2: row['Option2'],
            3: row['Option3'],
            4: row['Option4']
        }
        actual_answer = row['Answer']
        #print(actual_answer)

        # 1. 過濾掉 NaN 或空值的選項 (假設 options 是 dict 或 list)
        # 這裡用 str(v) != 'nan' 是為了處理 pandas 的情況
        valid_options = {k: v for k, v in options.items() if v and str(v).lower() != 'nan'}
        
        # 2. 動態生成選項字串
        options_text = "\n".join([f"{k}. {v}" for k, v in valid_options.items()])
        valid_numbers = ", ".join([str(k) for k in valid_options.keys()])

        # 建構 Prompt
        # 這裡的設計是為了讓 DeepSeek R1 能夠先思考再回答，並最後輸出固定格式
        prompt = f"""
Read the following article and answer the multiple-choice question.

Article:
{article}

Question:
{question}

Options:
{options_text}

Instruction:
Analyze the article and select the best option from the available choices listed above ({valid_numbers}). 
If an option is missing or empty, do not select it.

Finally, output the answer in the exact format: "Answer: X", where X is the option number.
Do not output anything else after the answer.
"""
        
        # 呼叫模型
        response_text = query_ollama(prompt, model=MODEL_NAME, think=THINK_MODE)
        
        # 解析答案
        predicted_answer = extract_answer(response_text)
        
        # 比對結果
        is_correct = False
        try:
            # 處理 Excel 讀入的答案可能是浮點數或字串的情況
            actual_answer_int = int(actual_answer)
            if predicted_answer is not None and predicted_answer == actual_answer_int:
                correct_count += 1
                is_correct = True
            #else:
                #print("Article:", article)
                #print("Question:", question)
                #print("Option:", "1.", row['Option1'], "2.", row['Option2'], "3.", row['Option3'], "4.", row['Option4'])
                #print("Answer:", actual_answer)
                #print("Predicted Answer:", predicted_answer)
                #print('-'*30)
        except ValueError:
            pass # 忽略無效的標準答案格式

        # 儲存詳細結果
        results.append({
            "ID": row['ID'],
            "Actual": actual_answer,
            "Predicted": predicted_answer,
            "Correct": is_correct,
            "Model_Response": response_text
        })

        #print(predicted_answer)
        #if (index+1) % 500 == 0:
            #print(correct_count, " / ", (index+1), " = ", 100.0*correct_count/(index+1), "%")
            

    # 3. 計算並輸出正確率
    accuracy = (correct_count / total_questions) * 100 if total_questions > 0 else 0
    print("\n" + "="*30)
    print(f"處理完成！")
    print(f"總題數: {total_questions}")
    print(f"答對題數: {correct_count}")
    print(f"正確率 (Accuracy): {accuracy:.2f}%")
    print("="*30)

    # 4. 將結果存檔
    output_file = "result_analysis.xlsx"
    pd.DataFrame(results).to_excel(output_file, index=False)
    print(f"詳細結果已儲存至 {output_file}")

In [9]:
MODEL_NAME = 'gemma3:12b'  # 模型名稱
THINK_MODE = False
main()

開始使用 gemma3:12b 處理 1436 個問題...


100%|██████████████████████████████████████████████████████████████████████████████| 1436/1436 [07:50<00:00,  3.05it/s]


處理完成！
總題數: 1436
答對題數: 1293
正確率 (Accuracy): 90.04%
詳細結果已儲存至 result_analysis.xlsx


In [10]:
MODEL_NAME = 'qwen3:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 qwen3:14b 處理 1436 個問題...


100%|██████████████████████████████████████████████████████████████████████████████| 1436/1436 [04:57<00:00,  4.83it/s]



處理完成！
總題數: 1436
答對題數: 1325
正確率 (Accuracy): 92.27%
詳細結果已儲存至 result_analysis.xlsx


In [11]:
MODEL_NAME = 'deepseek-r1:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 deepseek-r1:14b 處理 1436 個問題...


100%|██████████████████████████████████████████████████████████████████████████████| 1436/1436 [38:05<00:00,  1.59s/it]


處理完成！
總題數: 1436
答對題數: 1305
正確率 (Accuracy): 90.88%
詳細結果已儲存至 result_analysis.xlsx


In [8]:
MODEL_NAME = 'llama2:13b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 llama2:13b 處理 1436 個問題...


100%|████████████████████████████████████████████████████████████████████████████| 1436/1436 [1:16:35<00:00,  3.20s/it]


處理完成！
總題數: 1436
答對題數: 1049
正確率 (Accuracy): 73.05%
詳細結果已儲存至 result_analysis.xlsx


In [14]:
MODEL_NAME = 'phi3:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 phi3:14b 處理 1436 個問題...


100%|██████████████████████████████████████████████████████████████████████████████| 1436/1436 [04:28<00:00,  5.34it/s]



處理完成！
總題數: 1436
答對題數: 1255
正確率 (Accuracy): 87.40%
詳細結果已儲存至 result_analysis.xlsx


In [8]:
MODEL_NAME = 'phi4:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 phi4:14b 處理 1436 個問題...


100%|████████████████████████████████████████████████████████████████████████████| 1436/1436 [1:19:16<00:00,  3.31s/it]



處理完成！
總題數: 1436
答對題數: 1317
正確率 (Accuracy): 91.71%
詳細結果已儲存至 result_analysis.xlsx


In [ ]:
MODEL_NAME = 'phi4-reasoning:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()